In [9]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="llama-3.1-8b-instant",
)

In [10]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at locaiton"""
    return f"It's sunny in {location}"
                                                          

model_with_tools = model.bind_tools([get_weather])


In [12]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:

    print(f"Tool : {tool_call['name']}")
    print(f"Args : {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': 'rxp31z0fp', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 222, 'total_tokens': 236, 'completion_time': 0.024420354, 'completion_tokens_details': None, 'prompt_time': 0.014297195, 'prompt_tokens_details': None, 'queue_time': 0.044728125, 'total_time': 0.038717549}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019cc165-b98f-7b90-870a-4f7c32b9a8a5-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'rxp31z0fp', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 222, 'output_tokens': 14, 'total_tokens': 236}
Tool : get_weather
Args : {'location': 'Boston'}


### Tool Exceution Loops

In [14]:
# Step 1 : Model generates tool calls
messages = [{"role":"user","content":"What's the weather in Boston?"}]
ai_msg =model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2 : Execute tools and collect results 
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)


# Step 3 : Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# The current weather in Boston in 72F and sunny

In [15]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '486kzyzpn', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 221, 'total_tokens': 235, 'completion_time': 0.015424084, 'completion_tokens_details': None, 'prompt_time': 0.029823297, 'prompt_tokens_details': None, 'queue_time': 0.047547323, 'total_time': 0.045247381}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cc168-a076-7151-a152-b3bef7f38460-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '486kzyzpn', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 221, 'output_tokens': 14, 'total_tokens': 235}),
 ToolMessage(content="It's su